In [2]:
import logging

import torch

from nichejepa.masks.multigene import MaskCollator
from nichejepa.utils.config import create_params_from_YAML_wandb_config

In [3]:
world_size = 1
rank = 0

In [4]:
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)
params = create_params_from_YAML_wandb_config('../../../nichejepa/configs/test.yaml',
                                              logger)

INFO:root:Loaded parameters from YAML file.


In [7]:
mask = MaskCollator(
    n_targets=params["mask"]["n_targets"],
    n_contexts=params["mask"]["n_contexts"],
    target_mask_size=params["mask"]["target_mask_size"],
    context_mask_size=params["mask"]["context_mask_size"],
    seq_len_cell=params["data"]["seq_len_cell"],
    seq_len_neighborhood=params["data"]["seq_len_neighborhood"],
    has_cls=params["data"]["has_cls"])

In [8]:
non_zero_seq_len_cell = 10
non_zero_seq_len_neighborhood = 5

In [16]:
g = torch.Generator()
g.manual_seed(1)

target_mask, target_mask_complement = mask._sample_gene_mask(
    non_zero_seq_len_cell=non_zero_seq_len_cell,
    non_zero_seq_len_neighborhood=non_zero_seq_len_neighborhood,
    mask_size=params["mask"]["target_mask_size"],
    generator=g, # Use the same generator to ensure reproducibility
    mask_type='target')

print(target_mask)
print(target_mask_complement)

tensor([  0,   2,   3,   4,   5,   6,   7,   8,   9,  10, 201])
tensor([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [8]:
context_mask, context_mask_complement = mask._sample_gene_mask(
non_zero_seq_len_cell=non_zero_seq_len_cell,
                    non_zero_seq_len_neighborhood=non_zero_seq_len_neighborhood,
                    mask_size=params["mask"]["context_mask_size"],
                    generator=g, # Use the same generator to ensure reproducibility
                    valid_token_masks=[target_mask_complement],
                    mask_type='context')

print(context_mask)
print(context_mask_complement)

tensor([  0, 649, 650])
tensor([1, 1, 1,  ..., 1, 1, 1], dtype=torch.int32)


In [10]:
batch = [(torch.tensor([1087, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10] + [0] * 990 + [1, 2, 3, 4, 5] + [0] * 995), # input tokens
          torch.tensor([1] * 1001 + [2] * 1000), # input seg label
          "CT A"), # input cell type
        ]

collated_batch, collated_masks_context, collated_masks_target = mask(batch)
print(collated_batch)
print(collated_masks_context)
print(collated_masks_target)

[tensor([[1087,    1,    2,  ...,    0,    0,    0]]), tensor([[1, 1, 1,  ..., 2, 2, 2]]), ('CT A',)]
[tensor([[   0, 1559, 1560]])]
[tensor([[   0,    9,   10, 1001, 1002]]), tensor([[0, 4, 5, 6, 7]])]
